# **MAESTRÍA EN INTELIGENCIA ARTIFICIAL APLICADA**

# Análisis de Grandes Volúmenes de datos

## Instituto Tecnológico de Monterrey

## Actividad 4: Métricas de Calidad de Resultados

---

Alumno : Emanuel Flores Martínez

Matrícula: A01796497

---

Este notebook (**Actividad 4 — Métricas de Calidad de Resultados**) da continuidad
directa a la **Evidencia 1** del equipo y a la **Actividad 3 del Módulo 4**, donde se
caracterizó la población *P* (pares **BTCUSDT, ETHUSDT, XRPUSDT** del dataset
*Binance Full History*), se definió el particionamiento
`A = symbol × B = volatility_level × C = market_period` y se propuso el
**muestreo aleatorio estratificado** (`sampleBy`) para construir la muestra *M*.

A diferencia de la Actividad 3 —cuyo foco fue *aplicar* un algoritmo supervisado y uno
no supervisado—, el objetivo de esta actividad es **medir, bajo distintas métricas, la
calidad de los modelos** entrenados sobre grandes volúmenes de datos con **PySpark**.
Para ello se incorporan tres elementos nuevos respecto a la Actividad 3:

1. **Determinación estadística del tamaño de cada partición** `Mi` (fórmula de Cochran)
   para que la muestra sea representativa **sin inyectar sesgo de selección**.
2. **Ajuste de hiper-parámetros con validación cruzada** (`CrossValidator` +
   `ParamGridBuilder`) como técnica explícita **contra el sobre-ajuste**.
3. **Batería ampliada de métricas** (globales, por clase, AUC *one-vs-rest*, *log-loss*,
   detección de *overfitting* train-vs-test, métricas internas de *clustering*),
   seleccionadas considerando su **escalabilidad en un entorno distribuido**.

### Contenido de la actividad

| Sección | Contenido | Peso |
|---|---|---|
| **1** | Construcción de la muestra *M* (tamaño justificado + verificación de representatividad) | 20% |
| **2** | Construcción Train – Test (split estratificado, disjunto y sin sesgo) | 20% |
| **3** | Selección de métricas para medir la calidad de resultados | 20% |
| **4** | Entrenamiento de modelos de aprendizaje (con *tuning* y anti-sobreajuste) | 20% |
| **5** | Análisis de resultados (fortalezas y áreas de oportunidad) | 20% |

> **Algoritmos** (continuidad con la Actividad 3): **`RandomForestClassifier`**
> (supervisado, objetivo `market_direction`) y **`KMeans`** (no supervisado, regímenes
> de mercado). El énfasis de esta entrega está en **cómo se mide su calidad**, no en
> cambiar de algoritmo.

# 0. Preparación del entorno y carga de datos

Reutilizamos la conexión a Google Drive y el dataset *Binance Full History* de la
Evidencia 1. La población *P* completa contiene ~1000 pares; como en la Actividad 3,
trabajamos con los **tres pares de la población de estudio** (BTC/ETH/XRP), de los que
luego se extrae la muestra *M*. Los datos están almacenados en la nube (Drive) por su
volumen (varios GB en formato Parquet).

In [ ]:
import os, zipfile, fnmatch
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Kaggle credentials
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
!pip install -q kaggle

DATASET     = "jorijnsmit/binance-full-history"
WANTED      = ["BTC", "ETH", "XRP"]                                       # pairs vs USDT

EXTRACT_DIR = Path("/content/drive/MyDrive/binance_full_history/files")   # only 3 parquet, on Drive
TMP_DIR     = Path("/content/_binance_tmp")                               # ephemeral ZIP
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)

# 1. If the 3 parquet files already exist on Drive -> nothing to do
existing = sorted(EXTRACT_DIR.glob("*.parquet"))
def _norm(s):
    return ''.join(c for c in s.upper() if c.isalnum())
already = {b for b in WANTED if any(b in _norm(p.stem) and "USDT" in _norm(p.stem) for p in existing)}

if already == set(WANTED):
    print("The 3 parquet files are already on Drive - skipping download.")
    print("Files:", [p.name for p in existing])
else:
    missing = set(WANTED) - already
    print(f"Missing on Drive: {missing}. Downloading ZIP to ephemeral disk...")

    # 2. Download ZIP to /content (ephemeral, ~26 GB, fits on Colab's disk)
    zip_files = list(TMP_DIR.glob("*.zip"))
    if not zip_files:
        get_ipython().system(f'kaggle datasets download -d {DATASET} -p {TMP_DIR}')
        zip_files = list(TMP_DIR.glob("*.zip"))
    ZIP_PATH = zip_files[0]
    print(f"ZIP: {ZIP_PATH}  ({ZIP_PATH.stat().st_size / (1024**3):.2f} GB)")

    # 3. Extract ONLY the 3 pairs straight to Drive
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        names = z.namelist()
        for base in WANTED:
            matches = [n for n in names if fnmatch.fnmatch(n.upper(), f"*{base}*USDT*.PARQUET")]
            for n in matches:
                dest = EXTRACT_DIR / Path(n).name      # flatten path -> straight into EXTRACT_DIR
                with z.open(n) as src, open(dest, "wb") as out:
                    out.write(src.read())
                print("Extracted to Drive:", dest.name)

    # 4. Delete the ephemeral ZIP to free space
    ZIP_PATH.unlink()
    print("Ephemeral ZIP deleted.")

print("\nParquet files available on Drive:", sorted(p.name for p in EXTRACT_DIR.glob("*.parquet")))

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Binance ML - Activity 4 Quality Metrics")
    .config("spark.driver.memory", "10g")
    .config("spark.sql.files.maxPartitionBytes", "128MB")
    .config("spark.sql.shuffle.partitions", "64")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

In [ ]:
# Robust lookup of each pair's parquet file.
import glob

SYMBOLS = ["BTCUSDT", "ETHUSDT", "XRPUSDT"]

all_files = glob.glob(os.path.join(str(EXTRACT_DIR), "*.parquet"))
print(f"Total parquet files available: {len(all_files)}")

def find_pair_file(symbol):
    target = _norm(symbol)
    for f in all_files:
        if _norm(os.path.basename(f).replace('.parquet', '')) == target:
            return f
    return None

pair_files = {sym: find_pair_file(sym) for sym in SYMBOLS}
for sym, f in pair_files.items():
    print(f"{sym:>8} -> {f}")

missing_files = [s for s, f in pair_files.items() if f is None]
assert not missing_files, f"No parquet found for: {missing_files}."


In [ ]:
# Load the 3 pairs and union them, adding a 'symbol' column
from functools import reduce
from pyspark.sql.functions import lit

dfs = []
for sym in SYMBOLS:
    d = spark.read.parquet(pair_files[sym]).withColumn("symbol", lit(sym))
    dfs.append(d)

df3 = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)
df3.printSchema()
print("Columns:", df3.columns)

In [ ]:
# Row count per pair (load check)
from pyspark.sql.functions import count
df3.groupBy("symbol").agg(count("*").alias("rows")).show()

## 0.1 Variables derivadas de caracterización

Recreamos las variables derivadas de la Evidencia 1, base tanto del particionamiento
como de los modelos:

- `return_pct = (close - open) / open * 100`
- `intrabar_volatility_pct = (high - low) / open * 100`
- `taker_buy_ratio = taker_buy_base_asset_volume / volume` (presión compradora)
- `market_direction` ∈ {bullish, bearish, neutral} — **variable objetivo supervisada**
- `market_session` ∈ {Asia, Europe, America}
- `hour` (hora UTC de la vela)

In [ ]:
from pyspark.sql.functions import col, when, hour

df_feat = (
    df3
    .filter(col("open").isNotNull() & (col("open") > 0))
    .withColumn("return_pct", (col("close") - col("open")) / col("open") * 100)
    .withColumn("intrabar_volatility_pct", (col("high") - col("low")) / col("open") * 100)
    .withColumn(
        "taker_buy_ratio",
        when(col("volume") > 0, col("taker_buy_base_asset_volume") / col("volume")).otherwise(0.0),
    )
    .withColumn("hour", hour(col("open_time")))
    .withColumn(
        "market_direction",
        when(col("close") > col("open"), "bullish")
        .when(col("close") < col("open"), "bearish")
        .otherwise("neutral"),
    )
    .withColumn(
        "market_session",
        when(col("hour") < 8, "Asia")
        .when(col("hour") < 16, "Europe")
        .otherwise("America"),
    )
)

df_feat.select(
    "symbol", "open_time", "open", "close", "return_pct",
    "intrabar_volatility_pct", "taker_buy_ratio", "market_direction", "market_session"
).show(5, truncate=False)

## 0.2 Variables de partición `volatility_level` (B) y `market_period` (C)

Tal como en la Evidencia 1, el nivel de volatilidad se define **por símbolo** usando
percentiles de `intrabar_volatility_pct` (LOW ≤ p89, MEDIUM entre p89 y p99, HIGH > p99),
y el periodo de mercado por el año de la vela (EARLY < 2020, BULL 2020-2021,
MODERN ≥ 2022). Estas dos variables, junto con `symbol`, forman las **27 particiones**
`A × B × C`.

In [ ]:
# Per-symbol volatility thresholds (approxQuantile, relativeError=0.001)
thresholds = {}
for sym in SYMBOLS:
    q = (
        df_feat.filter(col("symbol") == sym)
        .approxQuantile("intrabar_volatility_pct", [0.89, 0.99], 0.001)
    )
    thresholds[sym] = q
    print(f"{sym}: p89={q[0]:.4f}%  p99={q[1]:.4f}%")

thr_rows = [(sym, float(thresholds[sym][0]), float(thresholds[sym][1])) for sym in SYMBOLS]
thr_df = spark.createDataFrame(thr_rows, ["symbol", "p89", "p99"])

In [ ]:
# Assign volatility_level (B) and market_period (C)
from pyspark.sql.functions import year

df_lvl = (
    df_feat.join(thr_df, on="symbol", how="left")
    .withColumn(
        "volatility_level",
        when(col("intrabar_volatility_pct") <= col("p89"), "LOW")
        .when(col("intrabar_volatility_pct") <= col("p99"), "MEDIUM")
        .otherwise("HIGH"),
    )
    .withColumn(
        "market_period",
        when(year(col("open_time")) < 2020, "EARLY")
        .when(year(col("open_time")) <= 2021, "BULL")
        .otherwise("MODERN"),
    )
    .drop("p89", "p99")
)

print("=== Distribution by volatility_level ===")
df_lvl.groupBy("symbol", "volatility_level").agg(count("*").alias("n")).orderBy("symbol", "volatility_level").show()
print("=== Distribution by market_period ===")
df_lvl.groupBy("market_period").agg(count("*").alias("n")).show()

# 1 Construcción de la muestra M

La población *P* son las ~7.9 millones de velas de 1 minuto de BTC/ETH/XRP. La muestra
*M* se construye por **muestreo aleatorio estratificado** sobre las **27 particiones**
`Mi = symbol × volatility_level × market_period`, de modo que
*M* = ⋃ *Mi* y cada *Mi* es una unidad de muestreo independiente.

A diferencia de la Actividad 3, aquí el número de instancias de cada *Mi* **no se fija
arbitrariamente**: se determina mediante un criterio estadístico que garantiza que *M*
sea representativa de *P* **sin inyectar sesgo de selección** que altere la calidad de
los resultados.

## 1.1 Determinación del tamaño de muestra (criterio estadístico)

Para que *M* refleje a *P* sin sesgo, el tamaño total `n` se calcula con la **fórmula de
Cochran para poblaciones finitas** (estimación de una proporción):

$$ n_0 = \frac{Z^2 \, p \, (1-p)}{e^2} \qquad n = \frac{n_0}{1 + \dfrac{n_0 - 1}{N}} $$

donde:
- `N` = tamaño de la población (filas de los 3 pares),
- `Z` = valor crítico normal (1.96 para **95 %** de confianza),
- `p` = proporción esperada; usamos `p = 0.5` porque **maximiza la varianza** y por
  tanto el tamaño requerido (escenario más conservador),
- `e` = margen de error admisible.

Calculamos el `n` **mínimo** que asegura representatividad y, por encima de él, fijamos
el tamaño operativo de *M* verificando el margen de error efectivo que se obtiene.

In [ ]:
from pyspark.sql.functions import concat_ws
import math

# Stratum key = partition (A x B x C)
df_strat = df_lvl.withColumn(
    "stratum", concat_ws("|", col("symbol"), col("volatility_level"), col("market_period"))
).cache()

# N = population size (the 3 pairs)
N = df_strat.count()
print(f"Population size N (3 pairs): {N:,}")

# --- Cochran's formula -------------------------------------------------------
Z = 1.96     # 95% confidence
p = 0.50     # max-variance (most conservative)
def cochran_n(N, e, Z=1.96, p=0.5):
    n0 = (Z**2 * p * (1 - p)) / (e**2)
    return n0 / (1 + (n0 - 1) / N)

for e in [0.01, 0.005, 0.0037]:
    n_req = cochran_n(N, e)
    print(f"  margin of error e={e*100:.2f}%  ->  required n = {math.ceil(n_req):,}")

# Minimum representative size at e = 1%
N_MIN = math.ceil(cochran_n(N, 0.01))
print(f"\nMinimum representative sample (95% conf., e=1%): n_min = {N_MIN:,}")

In [ ]:
# Operating size of M: we choose a target well above n_min for robustness in rare
# strata (HIGH) and for a representative 80/20 split. We then report the EFFECTIVE
# margin of error that this size achieves (it must be << 1%).
TARGET_M = 70_000

# Effective margin of error for the chosen n (finite-population correction)
def effective_error(N, n, Z=1.96, p=0.5):
    fpc = (N - n) / (N - 1)            # finite population correction
    return Z * math.sqrt(p * (1 - p) / n * fpc)

e_eff = effective_error(N, TARGET_M)
print(f"Chosen operating size for M : {TARGET_M:,} instances")
print(f"  -> {TARGET_M/N_MIN:.1f}x the statistical minimum (n_min={N_MIN:,})")
print(f"  -> effective margin of error at 95% conf.: e = {e_eff*100:.3f}%")
print("Interpretation: with this size, sample proportions deviate from the population")
print("by less than +/- {:.2f} percentage points with 95% confidence.".format(e_eff*100))

## 1.2 Asignación proporcional y muestreo estratificado

Para **no inyectar sesgo**, se usa **asignación proporcional**: cada partición *Mi*
recibe un número de instancias proporcional a su peso en la población,
`|Mi| ≈ n · (|Pi| / |P|)`. En `sampleBy` esto equivale a aplicar **la misma fracción de
muestreo a todos los estratos**. Así, la distribución conjunta
`symbol × volatility_level × market_period` de *M* **reproduce la de *P***, incluida la
rareza natural de los regímenes HIGH (≈ 1 %).

> **¿Por qué proporcional y no balanceado?** Un muestreo *balanceado* (igual `n` por
> estrato) sobre-representaría los regímenes raros e **inyectaría sesgo** en la
> probabilidad de ocurrencia de los patrones —justo lo que la actividad pide evitar—.
> La asignación proporcional preserva las probabilidades de *P*, condición necesaria
> para que las métricas de calidad de la sección 5 sean insesgadas.

In [ ]:
# Same fraction for every stratum  ==  proportional allocation
FRACTION = min(1.0, TARGET_M / N)
print(f"Proportional sampling fraction (all strata): {FRACTION:.6f}")

strata = [r["stratum"] for r in df_strat.select("stratum").distinct().collect()]
fractions = {s: FRACTION for s in strata}
print(f"Number of strata (present partitions Mi): {len(strata)}")

# Build M = U Mi  via stratified sampling
M = df_strat.sampleBy("stratum", fractions=fractions, seed=42).cache()
n_M = M.count()
print(f"\nSample M size: {n_M:,} instances")

## 1.3 Verificación de representatividad (control de sesgo)

Comprobamos empíricamente que *M* no introdujo sesgo: comparamos, estrato por estrato,
la **proporción en la población** `|Pi|/|P|` contra la **proporción en la muestra**
`|Mi|/|M|`. Si la asignación proporcional funcionó, ambas distribuciones deben
coincidir y la **desviación absoluta máxima** debe ser muy pequeña (del orden del margen
de error calculado en 1.1).

In [ ]:
from pyspark.sql.functions import round as sround

# Population proportions per stratum
pop_prop = (df_strat.groupBy("stratum").agg(count("*").alias("n_pop"))
            .withColumn("prop_pop", col("n_pop") / N))

# Sample proportions per stratum
samp_prop = (M.groupBy("stratum").agg(count("*").alias("n_samp"))
             .withColumn("prop_samp", col("n_samp") / n_M))

comp = (pop_prop.join(samp_prop, on="stratum", how="left").fillna(0)
        .withColumn("abs_diff_pct", sround((col("prop_samp") - col("prop_pop")) * 100, 4))
        .orderBy(col("abs_diff_pct").desc()))

print("=== Population vs Sample proportion per stratum (representativeness) ===")
comp.select("stratum", "n_pop", "n_samp",
            sround(col("prop_pop")*100, 3).alias("pop_%"),
            sround(col("prop_samp")*100, 3).alias("samp_%"),
            "abs_diff_pct").show(27, truncate=False)

max_dev = comp.agg({"abs_diff_pct": "max"}).collect()[0][0]
print(f"Maximum absolute deviation across the 27 strata: {abs(max_dev):.4f} percentage points")
print("(Should be of the same order as the margin of error from 1.1 -> no selection bias.)")

In [ ]:
# Composition of M by the main partition dimensions (sanity view)
print("=== M composition by symbol x volatility_level ===")
M.groupBy("symbol", "volatility_level").agg(count("*").alias("n")).orderBy("symbol", "volatility_level").show(30)

## 1.4 Limpieza y preparación de M

Antes de modelar, dejamos *M* lista: (1) diagnóstico de nulos, (2) casteo a tipos
numéricos `double`, y (3) tratamiento de *outliers* por **winsorización p1/p99** sobre
las variables de cola pesada (`volume`, `quote_asset_volume`, `number_of_trades`,
volúmenes *taker*) y recorte de `return_pct` a ±25 % (errores / *flash crashes*). La
winsorización evita que valores extremos distorsionen tanto al *clustering* (basado en
distancias) como al escalado.

In [ ]:
# 1.4.1 Null diagnosis in M
from pyspark.sql.functions import isnan

cols_check = ["open", "high", "low", "close", "volume", "quote_asset_volume",
              "number_of_trades", "taker_buy_base_asset_volume",
              "taker_buy_quote_asset_volume", "return_pct",
              "intrabar_volatility_pct", "taker_buy_ratio"]

null_counts = M.select([
    count(when(col(c).isNull() | isnan(col(c)), c)).alias(c) for c in cols_check
])
print("=== Null / NaN values per column (in M) ===")
null_counts.show(truncate=False, vertical=True)

In [ ]:
# 1.4.2 Null cleanup and type casting
from pyspark.sql.types import DoubleType

num_cols = ["volume", "quote_asset_volume", "number_of_trades",
            "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume",
            "return_pct", "intrabar_volatility_pct", "taker_buy_ratio"]

M_clean = M.dropna(subset=["close", "open", "volume", "number_of_trades"])
for c in num_cols:
    M_clean = M_clean.withColumn(c, col(c).cast(DoubleType()))

print(f"Rows after dropping critical nulls: {M_clean.count():,}")

In [ ]:
# 1.4.3 Outlier treatment (p1/p99 winsorization)
winsor_cols = ["volume", "quote_asset_volume", "number_of_trades",
               "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume"]

bounds = {}
for c in winsor_cols:
    lo, hi = M_clean.approxQuantile(c, [0.01, 0.99], 0.001)
    bounds[c] = (lo, hi)
    print(f"{c:>30}: p1={lo:,.4f}  p99={hi:,.4f}")

M_prep = M_clean
for c, (lo, hi) in bounds.items():
    M_prep = M_prep.withColumn(
        c, when(col(c) < lo, lo).when(col(c) > hi, hi).otherwise(col(c))
    )

M_prep = M_prep.withColumn(
    "return_pct",
    when(col("return_pct") > 25, 25.0).when(col("return_pct") < -25, -25.0).otherwise(col("return_pct")),
).cache()

print(f"\nPre-processed sample M: {M_prep.count():,} instances")
M_prep.select(num_cols).describe().show()

# 2 Construcción Train – Test

Asumimos `M = {Mi}` ya construida. Para obtener los conjuntos de entrenamiento `Tr` y
prueba `Ts` retomamos la **estrategia de muestreo del paso 4 de la Actividad 3**:
*split* aleatorio **estratificado**, ahora estratificando por **la partición `Mi`**
(`symbol × volatility_level × market_period`). Es decir, **cada `Mi` se divide
internamente** en `Tri` y `Tsi`, y luego `Tr = ⋃ Tri`, `Ts = ⋃ Tsi`.

**Por qué estratificar por `Mi`.** Un *split* puramente aleatorio podría, por azar,
sobre-representar un régimen (p. ej. dejar casi todas las velas HIGH en *train*),
sesgando la estimación de calidad. Estratificar por `Mi` **garantiza que la composición
de regímenes de *P* se conserva idéntica en *train* y *test***, de modo que la
probabilidad de ocurrencia de los patrones no se desvía en ninguno de los dos conjuntos.

**Porcentaje de división 80 / 20 — justificación.** Es el estándar más usado: con 80 %
el modelo dispone de suficientes ejemplos por estrato y por clase para aprender patrones
estables; el 20 % restante es un *test* lo bastante grande (varios miles de instancias)
para estimar la generalización con **baja varianza**. Dado el tamaño de *M*, el 20 %
sigue siendo representativo de cada `Mi`.

**Garantías exigidas.** Construimos el *split* con un identificador único y un
*anti-join*, de modo que se cumpla `Tri ∩ Tsi = ∅` (conjuntos disjuntos) y
`⋃(Tri ∪ Tsi) = M` (cobertura total).

In [ ]:
# 2.1 Stratified 80/20 split by partition Mi, with a unique id to guarantee disjoint sets
from pyspark.sql.functions import monotonically_increasing_id

TRAIN_FRAC = 0.8
M_id = M_prep.withColumn("row_id", monotonically_increasing_id()).cache()

# Stratify by the partition key 'stratum' (A x B x C) -> each Mi split 80/20
strata_now = [r["stratum"] for r in M_id.select("stratum").distinct().collect()]
split_fracs = {s: TRAIN_FRAC for s in strata_now}

train = M_id.sampleBy("stratum", fractions=split_fracs, seed=7).cache()
# test = M \ train  (anti-join by id => disjoint sets)
test = M_id.join(train.select("row_id"), on="row_id", how="left_anti").cache()

n_tr, n_ts = train.count(), test.count()
print(f"Train (Tr): {n_tr:,}   Test (Ts): {n_ts:,}")
print(f"Effective split: {n_tr/(n_tr+n_ts)*100:.2f}% / {n_ts/(n_tr+n_ts)*100:.2f}%")

In [ ]:
# 2.2 Formal checks: disjoint sets and full coverage
inter = train.select("row_id").intersect(test.select("row_id")).count()
union = n_tr + n_ts
print("=== Set guarantees ===")
print(f"  |Tr ∩ Ts|           = {inter}        (must be 0  -> disjoint)")
print(f"  |Tr| + |Ts|         = {union:,}")
print(f"  |M|                 = {M_id.count():,}")
print(f"  Coverage Tr ∪ Ts==M : {union == M_id.count()}")

In [ ]:
# 2.3 Bias check: stratum and target proportions preserved in Tr and Ts
print("=== market_direction distribution (target) ===")
for name, dset in [("TRAIN", train), ("TEST", test)]:
    tot = dset.count()
    print(f"\n{name} (n={tot:,})")
    (dset.groupBy("market_direction").agg(count("*").alias("n"))
         .withColumn("pct", sround(col("n")/tot*100, 2))
         .orderBy("market_direction").show())

print("=== volatility_level distribution (partition dimension B) ===")
for name, dset in [("TRAIN", train), ("TEST", test)]:
    tot = dset.count()
    print(f"\n{name}")
    (dset.groupBy("volatility_level").agg(count("*").alias("n"))
         .withColumn("pct", sround(col("n")/tot*100, 2))
         .orderBy("volatility_level").show())

# 3 Selección de métricas para medir calidad de resultados

La elección de métricas se hace **antes** de experimentar y atendiendo a dos
condicionantes: (a) la **naturaleza de cada tarea** (clasificación supervisada vs.
*clustering* no supervisado) y (b) que trabajamos con **grandes volúmenes de datos**, lo
que obliga a usar métricas **calculables de forma distribuida**.

## 3.1 Consideraciones para Big Data

En un entorno distribuido (Spark) no toda métrica es igual de viable:

- **Escalabilidad / cómputo distribuido.** Priorizamos métricas que Spark MLlib evalúa
  *en el clúster* sin traer los datos al *driver*. Los evaluadores
  `MulticlassClassificationEvaluator`, `BinaryClassificationEvaluator` y
  `ClusteringEvaluator` operan sobre el `DataFrame` distribuido (agregaciones tipo
  *map-reduce*), evitando un `collect()` masivo que desbordaría la memoria del *driver*.
- **Coste de ordenamiento.** Métricas basadas en ranking (AUC-ROC, AUC-PR) requieren
  *ordenar* las predicciones: son más costosas (*shuffle*) que `accuracy`/`F1`, que solo
  necesitan conteos. Las usamos de forma dirigida, no indiscriminada.
- **Aproximaciones distribuidas.** El **silhouette** de Spark usa la formulación
  *silhouette* con distancia al cuadrado computada de forma distribuida (Karthik et al.),
  apta para millones de puntos; el silhouette clásico O(n²) sería inviable.
- **Robustez al desbalance.** Con clases desbalanceadas (`neutral` ≈ 5 %), una sola
  métrica global puede engañar; se complementa con métricas **por clase**.

## 3.2 Métricas para el modelo supervisado (clasificación)

| Métrica | Qué mide | Por qué la usamos |
|---|---|---|
| **Accuracy** | % de aciertos global | Referencia rápida; **insuficiente** sola por el desbalance. |
| **F1 ponderado** | media armónica precision-recall, ponderada por soporte | Métrica **principal**: equilibra precision y recall y pondera por tamaño de clase. |
| **Precision / Recall ponderados** | exactitud y cobertura globales | Descomponen el F1; informan si el error es por falsos positivos o negativos. |
| **Precision / Recall / F1 por clase** (`*ByLabel`) | desempeño en cada clase | Detectan si la clase minoritaria (`neutral`) se predice mal pese a un buen global. |
| **Matriz de confusión** | conteo real vs predicho | Diagnóstico fino: revela *qué* clases se confunden entre sí. |
| **AUC-ROC (one-vs-rest)** | separabilidad clase vs resto, independiente del umbral | Mide la **calidad del ranking de probabilidades**, no solo la decisión dura. |
| **AUC-PR (one-vs-rest)** | área precision-recall | Más informativa que ROC para **clases minoritarias** (neutral). |
| **Log-loss** | calibración de las probabilidades | Penaliza predicciones confiadas y equivocadas; evalúa la **probabilidad**, no solo la etiqueta. |

**Métrica de selección de modelo:** **F1 ponderado** (la que optimiza el
`CrossValidator` de la sección 4), por ser la más equilibrada ante el desbalance.
La **matriz de confusión** y las métricas **por clase** se usan para el análisis
cualitativo de la sección 5.

> **Nota sobre AUC en multiclase.** Spark calcula AUC para problemas binarios. Como
> `market_direction` tiene 3 clases, computamos el AUC con la estrategia **one-vs-rest**:
> por cada clase generamos un problema binario (clase vs. resto) usando la probabilidad
> que el modelo asignó a esa clase, y promediamos (macro). Es la práctica estándar para
> extender AUC a multiclase.

## 3.3 Métricas para el modelo no supervisado (clustering)

Al no haber etiqueta de referencia, la calidad se mide con **criterios internos**:

| Métrica | Qué mide | Por qué la usamos |
|---|---|---|
| **Coeficiente de silueta** | cohesión intra-cluster vs separación inter-cluster, en [-1, 1] | Métrica **principal** para elegir `k`; cuanto más alto, mejor definidos los grupos. Spark la calcula de forma distribuida. |
| **WSSSE / inercia** | suma de distancias al cuadrado a cada centroide | Mide compacidad; base del **método del codo**. Decrece siempre con `k`, por eso es criterio **secundario**. |
| **Tamaño y perfil de clusters** | nº de instancias y medias por grupo | Verifica que los grupos sean **no triviales** e **interpretables** (no un cluster que se queda con todo). |

> **Davies-Bouldin / Calinski-Harabasz** (no nativas en Spark MLlib) se mencionan como
> complementos teóricos; para mantener todo el cómputo distribuido nos apoyamos en
> **silhouette + WSSSE**, ambas escalables y nativas en `pyspark.ml`.

In [ ]:
# 3.4 Reusable evaluation helpers (used in section 5).
# All of them operate on the distributed DataFrame (no driver-side collect of the data).
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

def classification_metrics(pred_df, labels, label_col="label",
                           pred_col="prediction", prob_col="probability"):
    'Return global + per-class metrics for a multiclass classifier.'
    out = {}
    for m in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
        out[m] = MulticlassClassificationEvaluator(
            labelCol=label_col, predictionCol=pred_col, metricName=m).evaluate(pred_df)
    # log-loss (needs probabilities)
    try:
        out["logLoss"] = MulticlassClassificationEvaluator(
            labelCol=label_col, predictionCol=pred_col, probabilityCol=prob_col,
            metricName="logLoss").evaluate(pred_df)
    except Exception as ex:
        out["logLoss"] = None
    # per-class precision / recall / f1
    per_class = {}
    for idx, name in enumerate(labels):
        pc = {}
        for m in ["precisionByLabel", "recallByLabel", "fMeasureByLabel"]:
            pc[m] = MulticlassClassificationEvaluator(
                labelCol=label_col, predictionCol=pred_col,
                metricName=m, metricLabel=float(idx)).evaluate(pred_df)
        per_class[name] = pc
    return out, per_class

def auc_one_vs_rest(pred_df, labels, label_col="label", prob_col="probability"):
    'Macro AUC-ROC / AUC-PR via one-vs-rest using class probabilities.'
    d = pred_df.withColumn("__p", vector_to_array(col(prob_col)))
    rows = []
    for idx, name in enumerate(labels):
        dd = (d.withColumn("__bin", (col(label_col) == float(idx)).cast("double"))
                .withColumn("__pos", col("__p")[idx]))
        roc = BinaryClassificationEvaluator(labelCol="__bin", rawPredictionCol="__pos",
                                            metricName="areaUnderROC").evaluate(dd)
        pr = BinaryClassificationEvaluator(labelCol="__bin", rawPredictionCol="__pos",
                                           metricName="areaUnderPR").evaluate(dd)
        rows.append((name, roc, pr))
    return rows

print("Evaluation helpers ready: classification_metrics(), auc_one_vs_rest()")

# 4 Entrenamiento de Modelos de Aprendizaje

**Estrategia general.** Se construye un *pipeline* reproducible (indexado →
codificación → ensamblado de *features* → modelo). El procesamiento de datos es
**distribuido** (todo se ejecuta sobre los `DataFrame` de Spark) y el ajuste de
hiper-parámetros se hace con **validación cruzada k-fold**.

**Estrategia anti-sobreajuste.** Para impedir que los modelos queden sobre-ajustados se
combinan cuatro mecanismos:
1. **Validación cruzada (`CrossValidator`, k=3)** + **`ParamGridBuilder`**: los
   hiper-parámetros se eligen por desempeño en *folds* de validación, no en *train*.
2. **Control de complejidad**: se acota `maxDepth` (profundidad del árbol) en el grid;
   árboles menos profundos generalizan mejor.
3. **Ensamble con submuestreo**: Random Forest promedia muchos árboles entrenados sobre
   *bootstraps* distintos, lo que reduce la varianza por diseño.
4. **Verificación train-vs-test** (sección 5): se comparan las métricas en ambos
   conjuntos; una brecha grande señalaría sobre-ajuste.

**Prevención de fuga de información (*data leakage*).** Para el modelo supervisado se
**excluyen** `open`, `high`, `low`, `close` y `return_pct`, porque `market_direction` se
deriva del signo de `(close − open)`; usarlas haría que el modelo "hiciera trampa". El
problema planteado es genuino: *¿puede predecirse la dirección de la vela a partir de su
microestructura de mercado (volumen, nº de trades, presión compradora, volatilidad)?*

## 4.1 Modelo supervisado — `RandomForestClassifier` con validación cruzada

**Objetivo:** `market_direction` (bullish / bearish / neutral).
**Features:** numéricas (`volume`, `quote_asset_volume`, `number_of_trades`, volúmenes
*taker*, `taker_buy_ratio`, `intrabar_volatility_pct`, `hour`) + categóricas
*one-hot* (`symbol`, `market_session`).
**Pipeline:** `StringIndexer` → `OneHotEncoder` → `VectorAssembler` →
`RandomForestClassifier`.
**Grid de hiper-parámetros:** `numTrees ∈ {50, 100}` × `maxDepth ∈ {5, 10}`, evaluado por
**F1 ponderado** con **3-fold CV** (12 entrenamientos).

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier

numeric_features = ["volume", "quote_asset_volume", "number_of_trades",
                    "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume",
                    "taker_buy_ratio", "intrabar_volatility_pct", "hour"]
cat_cols = ["symbol", "market_session"]

label_indexer = StringIndexer(inputCol="market_direction", outputCol="label", handleInvalid="keep")
cat_indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep") for c in cat_cols]
cat_encoder = OneHotEncoder(inputCols=[c + "_idx" for c in cat_cols],
                            outputCols=[c + "_oh" for c in cat_cols])
assembler = VectorAssembler(
    inputCols=numeric_features + [c + "_oh" for c in cat_cols],
    outputCol="features",
)
rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42)

pipeline = Pipeline(stages=[label_indexer] + cat_indexers + [cat_encoder, assembler, rf])
print("Supervised pipeline defined.")

In [ ]:
# 4.1.1 Hyper-parameter tuning with k-fold cross-validation (anti-overfitting)
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
import numpy as np

paramGrid = (ParamGridBuilder()
             .addGrid(rf.numTrees, [50, 100])
             .addGrid(rf.maxDepth, [5, 10])
             .build())

cv_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                                 metricName="f1")
cv = CrossValidator(estimator=pipeline,
                    estimatorParamMaps=paramGrid,
                    evaluator=cv_evaluator,
                    numFolds=3,
                    parallelism=2,
                    seed=42)

print(f"Running 3-fold CV over {len(paramGrid)} hyper-parameter combinations "
      f"({len(paramGrid)*3} fits)... this may take several minutes.")
cv_model = cv.fit(train)
print("Cross-validation finished.")

In [ ]:
# 4.1.2 Cross-validation results and best hyper-parameters
avg = cv_model.avgMetrics
print("=== Mean validation F1 per hyper-parameter combination ===")
for pm, score in zip(cv_model.getEstimatorParamMaps(), avg):
    cfg = {p.name: v for p, v in pm.items()}
    print(f"  numTrees={cfg.get('numTrees'):>3}  maxDepth={cfg.get('maxDepth'):>2}  ->  CV F1 = {score:.4f}")

best_idx = int(np.argmax(avg))
best_cfg = {p.name: v for p, v in cv_model.getEstimatorParamMaps()[best_idx].items()}
print(f"\nBest combination: {best_cfg}   (CV F1 = {avg[best_idx]:.4f})")

rf_model = cv_model.bestModel          # best PipelineModel (refit on all train)
print("Best model selected and refit on the full training set.")

## 4.2 Modelo no supervisado — `KMeans` con selección de k

**Objetivo:** descubrir **regímenes de mercado** (grupos de velas con comportamiento
similar) sin usar etiquetas.
**Features de agrupamiento:** `volume`, `number_of_trades`, `return_pct`,
`intrabar_volatility_pct`, `taker_buy_ratio`. Como K-Means usa distancia euclidiana, es
**imprescindible estandarizar** (`StandardScaler`, media 0 / desv. 1); de lo contrario
`volume` dominaría por su escala.
**Selección de k:** probamos `k = 2…6` y elegimos el de mayor **silueta** (criterio
principal), apoyándonos en el **codo** del WSSSE como criterio secundario.

In [ ]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

cluster_features = ["volume", "number_of_trades", "return_pct",
                    "intrabar_volatility_pct", "taker_buy_ratio"]

assembler_c = VectorAssembler(inputCols=cluster_features, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

prep_pipeline = Pipeline(stages=[assembler_c, scaler])
prep_model = prep_pipeline.fit(M_prep)
data_scaled = prep_model.transform(M_prep).cache()
print("Scaled data ready:", data_scaled.count(), "instances")

In [ ]:
# 4.2.1 Choosing k via silhouette (+ WSSSE elbow)
evaluator = ClusteringEvaluator(featuresCol="features", predictionCol="prediction",
                                metricName="silhouette")
results = []
for k in range(2, 7):
    km = KMeans(featuresCol="features", predictionCol="prediction", k=k, seed=42)
    model_k = km.fit(data_scaled)
    pred_k = model_k.transform(data_scaled)
    sil = evaluator.evaluate(pred_k)
    wssse = model_k.summary.trainingCost
    results.append((k, sil, wssse))
    print(f"k={k}:  silhouette={sil:.4f}   WSSSE(inertia)={wssse:,.1f}")

best_k = max(results, key=lambda r: r[1])[0]
print(f"\nBest k by silhouette: k = {best_k}")

In [ ]:
# 4.2.2 Final K-Means model
kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=best_k, seed=42)
kmeans_model = kmeans.fit(data_scaled)
clustered = kmeans_model.transform(data_scaled).cache()

print(f"=== K-Means with k={best_k} ===")
print("Size of each cluster:")
clustered.groupBy("cluster").agg(count("*").alias("n")).orderBy("cluster").show()

# 5 Análisis de resultados

Aplicamos la batería de métricas seleccionada en la **sección 3** sobre los modelos
entrenados en la **sección 4**, y discutimos fortalezas y áreas de oportunidad.

## 5.1 Evaluación del modelo supervisado

In [ ]:
# 5.1.1 Predictions on the held-out test set + global / per-class metrics
labels_order = rf_model.stages[0].labels   # index -> class name
print("Label order (index -> class):", dict(enumerate(labels_order)))

pred_test = rf_model.transform(test)
glob, per_class = classification_metrics(pred_test, labels_order)

print("\n=== Global metrics on TEST ===")
for k, v in glob.items():
    print(f"  {k:>18}: {v:.4f}" if v is not None else f"  {k:>18}: n/a")

print("\n=== Per-class metrics on TEST ===")
for name, pc in per_class.items():
    print(f"  [{name}]  precision={pc['precisionByLabel']:.4f}  "
          f"recall={pc['recallByLabel']:.4f}  f1={pc['fMeasureByLabel']:.4f}")

In [ ]:
# 5.1.2 Confusion matrix (distributed groupBy/pivot)
print("=== Confusion matrix (rows = actual, columns = predicted) ===")
pred_test.groupBy("label").pivot("prediction").count().orderBy("label").show()
print("Index -> class:", dict(enumerate(labels_order)))

In [ ]:
# 5.1.3 AUC-ROC and AUC-PR (one-vs-rest, macro)
auc_rows = auc_one_vs_rest(pred_test, labels_order)
print("=== AUC per class (one-vs-rest) ===")
roc_vals, pr_vals = [], []
for name, roc, pr in auc_rows:
    print(f"  [{name}]  AUC-ROC={roc:.4f}   AUC-PR={pr:.4f}")
    roc_vals.append(roc); pr_vals.append(pr)
print(f"\n  Macro AUC-ROC = {sum(roc_vals)/len(roc_vals):.4f}")
print(f"  Macro AUC-PR  = {sum(pr_vals)/len(pr_vals):.4f}")

In [ ]:
# 5.1.4 Overfitting check: compare TRAIN vs TEST metrics
pred_train = rf_model.transform(train)
glob_tr, _ = classification_metrics(pred_train, labels_order)

print("=== Train vs Test (overfitting diagnosis) ===")
print(f"  {'metric':>18} | {'TRAIN':>8} | {'TEST':>8} | {'gap':>7}")
for m in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    gap = glob_tr[m] - glob[m]
    print(f"  {m:>18} | {glob_tr[m]:8.4f} | {glob[m]:8.4f} | {gap:7.4f}")
print("\nA small train-test gap indicates the cross-validated model is NOT overfitting.")

In [ ]:
# 5.1.5 Feature importances
rf_stage = rf_model.stages[-1]
imp = rf_stage.featureImportances.toArray()
print("=== Feature importances (Random Forest) ===")
for i, name in enumerate(numeric_features):
    print(f"  {name:>30}: {imp[i]:.4f}")
print("  (remaining positions correspond to the one-hot categoricals: symbol, market_session)")

### Discusión — modelo supervisado

**Desempeño global.** Sobre el conjunto de prueba (14 075 instancias) el mejor modelo
—`numTrees=100`, `maxDepth=5`, elegido por validación cruzada— obtuvo
**accuracy = 0.641**, **F1 ponderado = 0.639** (precision ponderada 0.647) y
**log-loss = 0.744**. El *baseline* de predecir siempre la clase mayoritaria
(*bullish*, 47.4 % del test) daría accuracy ≈ 0.474, por lo que el modelo **supera al
azar informado en ~16.7 puntos**: la microestructura de mercado **sí contiene señal**
sobre la dirección de la vela, aunque con un techo claro.

**Ausencia de sobre-ajuste.** La brecha entre *train* y *test* es **mínima**
(F1: 0.642 vs 0.639, *gap* = 0.0022; accuracy *gap* = 0.0013; la precision ponderada es
incluso ligeramente mayor en test). Además, la validación cruzada **prefirió
`maxDepth=5` sobre `maxDepth=10`** (que daba un F1 prácticamente igual o menor,
0.6396–0.6397): árboles más profundos no aportan capacidad útil, señal inequívoca de que
el modelo está **bien regularizado** y de que existe un **límite intrínseco de
predictibilidad** (~64 %) al usar solo información intra-vela. Las técnicas
anti-sobreajuste de la sección 4 cumplieron su función.

**Análisis por clase (matriz de confusión).** El *recall* por clase es:
- *bullish*: 4 389 / 6 670 ≈ **66 %**
- *bearish*: 4 375 / 6 686 ≈ **65 %**
- *neutral*: 263 / 719 ≈ **37 %**

El **error dominante es confundir *bullish* con *bearish*** (2 251 + 2 292 ≈ 4 543 casos
cruzados), algo esperable porque ambas dependen de un balance fino de presión
compradora/vendedora. La clase **neutral** muestra un patrón revelador: **precision alta
(0.84) pero recall bajo (0.37)**. Es decir, el modelo es **conservador** al predecir
neutral —cuando lo hace, acierta el 84 %—, pero se le escapa el 63 % de las velas
neutrales reales, que reparte entre bullish/bearish. Tiene sentido económico: las velas
neutrales (`close == open`) son planas y carecen de un patrón de volumen distintivo.

**Capacidad de *ranking* (AUC *one-vs-rest*).** El **macro AUC-ROC = 0.765** y el
**macro AUC-PR = 0.623** confirman una capacidad de discriminación moderada-buena por
encima del azar (ROC = 0.5). El dato más interesante es la clase **neutral**: su
**AUC-ROC = 0.869 es la más alta de las tres**, mientras que su **AUC-PR = 0.517 es la
más baja**. La lectura es clara: el modelo **sí separa bien las neutrales en el ranking
de probabilidades**, pero (i) la decisión dura por `argmax` sacrifica su *recall*, y
(ii) el fuerte desbalance (≈ 5 %) hunde la precision-recall. Esto sugiere que **ajustar
el umbral de decisión** (en lugar de `argmax`) recuperaría buena parte del *recall* de
neutral sin reentrenar.

**Variables más informativas.** La importancia confirma la hipótesis de microestructura:
`taker_buy_ratio` **domina con 0.585**, seguida de `intrabar_volatility_pct` (**0.186**)
y `number_of_trades` (**0.073**); el resto aporta < 0.05 y la hora del día es
prácticamente irrelevante (`hour` = 0.001). Es decir, **la proporción de volumen agresor
comprador es, con diferencia, el predictor más fuerte de la dirección**, coherente con la
teoría: el flujo de órdenes agresoras es lo que mueve el precio.

## 5.2 Evaluación del modelo no supervisado

In [ ]:
# 5.2.1 Internal clustering metrics for the final model
final_eval_sq = ClusteringEvaluator(featuresCol="features", predictionCol="cluster",
                                    metricName="silhouette", distanceMeasure="squaredEuclidean")
sil_final = final_eval_sq.evaluate(clustered)
wssse_final = kmeans_model.summary.trainingCost
print(f"=== Final clustering quality (k={best_k}) ===")
print(f"  Silhouette (squaredEuclidean): {sil_final:.4f}")
print(f"  WSSSE (inertia)              : {wssse_final:,.1f}")

In [ ]:
# 5.2.2 Cluster profiles (means in original units) + symbol composition
from pyspark.sql.functions import avg

print("=== Profile of each cluster (means in original units) ===")
(clustered.groupBy("cluster")
    .agg(
        sround(avg("volume"), 2).alias("volume"),
        sround(avg("number_of_trades"), 1).alias("n_trades"),
        sround(avg("return_pct"), 4).alias("return_pct"),
        sround(avg("intrabar_volatility_pct"), 4).alias("volat_pct"),
        sround(avg("taker_buy_ratio"), 3).alias("buy_ratio"),
        count("*").alias("n"),
    )
    .orderBy("cluster")
    .show(truncate=False))

print("=== Symbol composition within each cluster ===")
clustered.groupBy("cluster").pivot("symbol").count().orderBy("cluster").show()

### Discusión — modelo no supervisado

**Selección de k.** El coeficiente de silueta fue 0.26 (k=2), 0.34 (k=3), **0.71
(k=4)**, 0.43 (k=5) y 0.42 (k=6). El **máximo es inequívoco en k = 4**. La inercia
(WSSSE) decrece de forma monótona con k (299 k → 240 k → 221 k → 176 k → 163 k), por lo
que **no sirve por sí sola** para elegir k; se usa como apoyo (codo) y la **silueta como
criterio principal**. Un silhouette de **0.71 indica grupos bien definidos**, compactos
y separados entre sí.

**Regímenes de mercado encontrados (k = 4).** Interpretando los centroides en unidades
originales:

| Cluster | n | volumen | n_trades | return % | volat % | buy_ratio | Interpretación |
|---|---|---|---|---|---|---|---|
| **0** | 62 137 (88.4 %) | 32 k | 261 | ≈ 0 | 0.12 | 0.50 | **Mercado en calma**: baja actividad y volatilidad, dirección neutra. Es el régimen normal. |
| **1** | 4 341 (6.2 %) | 1.1 k | **2 699** | −0.05 | 0.27 | 0.49 | **Alta fragmentación**: muchísimas operaciones pequeñas con volumen bajo. |
| **2** | 2 538 (3.6 %) | **862 k** | 889 | −0.09 | 0.49 | 0.48 | **Picos de liquidez**: volumen extremo y volatilidad alta. |
| **3** | 1 288 (1.8 %) | 118 k | 863 | **+0.50** | **0.79** | **0.61** | **Rallies alcistas**: único retorno positivo, máxima volatilidad y máxima presión compradora. |

**Composición por símbolo (coherencia con la Evidencia 1).** Los grupos se alinean con
la caracterización por activo:
- El régimen de **picos de liquidez (cluster 2) es 98.5 % XRP** (2 500 de 2 538),
  confirmando que XRP es el activo de mayor volumen en unidades base (por su bajo precio
  unitario, una misma operación en dólares mueve muchísimas unidades).
- El de **alta fragmentación (cluster 1) está dominado por BTC** (3 239 de 4 341, 74.6 %),
  consistente con su microestructura de muchas órdenes pequeñas.
- El régimen de **calma (cluster 0)** reparte los tres pares de forma equilibrada
  (BTC 21 008, ETH 22 857, XRP 18 272), como corresponde al estado "normal" común a todos.
- Los **rallies (cluster 3)** mezclan los tres activos, siendo un fenómeno de mercado
  transversal más que propio de un par.

**Lectura final.** K-Means logró separar **estados de mercado con sentido económico**
(calma, fragmentación, picos de liquidez y rallies alcistas) **sin usar ninguna
etiqueta**, y los grupos resultan consistentes con la caracterización por activo de la
Evidencia 1. La alta silueta (0.71) da validez interna al resultado.

## 5.3 Síntesis: fortalezas y áreas de oportunidad

**Fortalezas**

- **Muestra *M* representativa y sin sesgo, demostrado con cifras.** Las 70 304
  instancias equivalen a **7.3× el mínimo estadístico** de Cochran (9 593) y alcanzan un
  **margen de error efectivo de 0.37 %**. La verificación P-vs-M arrojó una **desviación
  máxima de solo 0.106 puntos porcentuales** entre las proporciones de los 27 estratos,
  del mismo orden que el margen teórico → **sesgo de selección despreciable**.
- ***Split* train-test garantizado.** Conjuntos **disjuntos** (|Tr ∩ Ts| = 0) y de
  **cobertura total** (56 229 + 14 075 = 70 304 = |M|), con las proporciones de *target*
  (neutral 4.75 % en train vs 5.11 % en test) y de volatilidad preservadas en ambos.
- **Batería de métricas que aporta valor real.** Las métricas elegidas **revelaron lo que
  la accuracy global ocultaba**: pese a un 0.64 de accuracy, el *recall* de neutral es
  solo 0.37 y su AUC-ROC 0.87 → diagnóstico imposible con una sola métrica.
- **Modelos sin sobre-ajuste.** La validación cruzada eligió la configuración más simple
  (`maxDepth=5`) y la brecha train-test es < 0.003 en todas las métricas.
- **Clustering de calidad e interpretable** (silhouette 0.71, 4 regímenes con lectura
  económica clara).

**Áreas de oportunidad**

- **Clase *neutral* sub-detectada** (*recall* 0.37). Dado que su **AUC-ROC es 0.87**, el
  modelo la separa bien internamente; conviene **ajustar el umbral de decisión** o aplicar
  **pesos de clase / sobre-muestreo** para convertir esa capacidad de *ranking* en aciertos.
- **Techo de predictibilidad (~64 %) en la dirección.** Para superarlo habría que
  **enriquecer las *features*** con variables temporales rezagadas (memoria del mercado),
  información de *order-book* o indicadores técnicos, **siempre evitando fuga de
  información**.
- **Ampliar el espacio de modelos.** Como `maxDepth=5` venció a 10, explorar profundidades
  aún menores y `minInstancesPerNode`, y comparar contra **`GBTClassifier`**.
- **Refinar el *clustering*.** El cluster 0 concentra el 88 % de las velas; aplicar
  **`GaussianMixture`** (clusters elípticos con solape) o un sub-agrupamiento dentro del
  régimen de calma podría descubrir estructura más fina.

# 6 Conclusiones

- Se ejecutó el flujo completo de **medición de calidad** sobre ~7.9 millones de velas de
  Binance en PySpark: muestra *M* con **tamaño estadísticamente justificado** (Cochran,
  margen de error 0.37 %, desviación P-vs-M de 0.11 pp), ***split* estratificado
  garantizado** (disjunto y de cobertura total), **selección argumentada de métricas**
  pensadas para Big Data, entrenamiento con **ajuste de hiper-parámetros por validación
  cruzada** y una evaluación amplia (global, por clase, AUC *one-vs-rest*, *log-loss* y
  diagnóstico de *overfitting*).
- **Resultados clave.** El modelo **supervisado** alcanzó F1 = 0.639 (**+16.7 puntos**
  sobre el *baseline*), **sin sobre-ajuste** (brecha train-test < 0.003) y con
  `taker_buy_ratio` como **predictor dominante** (importancia 0.585). El modelo **no
  supervisado** obtuvo un **silhouette de 0.71** con `k = 4` y produjo **cuatro regímenes
  de mercado económicamente interpretables** (calma, fragmentación, picos de liquidez y
  rallies), coherentes con la caracterización por activo de la Evidencia 1.
- **Aprendizaje metodológico.** El valor de esta actividad no está en el algoritmo sino en
  **cómo se mide su calidad**: las métricas adecuadas (por clase, AUC, train-vs-test)
  **revelan matices que una métrica global oculta** (p. ej. el bajo *recall* de neutral
  pese a una buena accuracy), y la **validación cruzada** es lo que permite **confiar** en
  que los resultados generalizan sobre grandes volúmenes de datos.
- El diseño mantiene **continuidad** con la Evidencia 1 y la Actividad 3 (mismas variables
  de caracterización, mismo particionamiento A×B×C, misma familia de técnicas de muestreo).